# 05 Zero-shot: Qwen2.5-Coder-0.5B dan CodeT5-small tanpa fine-tuning
Qwen dievaluasi pada 4 set data x 2 mode (FULL, QONLY), beam 10, maks 30 token baru, kode dipotong 350 token.
CodeT5-small pretrained dievaluasi pada CodeQA Python test (FULL). Catatan: hasil Qwen QONLY Python pernah terkena bug pemrosesan keluaran
(lihat runs/zs_qwen_pythontest_qonly_old).

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## zeroshot_qwen.py

In [ ]:
# Default 'smoke' (32 contoh).
VARIANTS = {
    'smoke': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/python/test', '--mode', 'full', '--run-tag', 'smoke_zs_qwen_pythontest_full', '--limit', '32'],
    'pythontest_full': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/python/test', '--mode', 'full', '--run-tag', 'zs_qwen_pythontest_full'],
    'pythontest_qonly': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/python/test', '--mode', 'qonly', '--run-tag', 'zs_qwen_pythontest_qonly'],
    'javatest_full': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java/test', '--mode', 'full', '--run-tag', 'zs_qwen_javatest_full', '--batch', '4'],
    'javatest_qonly': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/kc-codeqa-reproduce/data/java/test', '--mode', 'qonly', '--run-tag', 'zs_qwen_javatest_qonly', '--batch', '4'],
    'classevalqa_full': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/classevalqa', '--mode', 'full', '--run-tag', 'zs_qwen_classevalqa_full', '--batch', '4'],
    'classevalqa_qonly': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/classevalqa', '--mode', 'qonly', '--run-tag', 'zs_qwen_classevalqa_qonly', '--batch', '4'],
    'cs1qa_full': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/cs1qa', '--mode', 'full', '--run-tag', 'zs_qwen_cs1qa_full', '--batch', '4'],
    'cs1qa_qonly': ['--data', '/home/nafisnaufal1426/inahuy/kacuy/ext_data/cs1qa', '--mode', 'qonly', '--run-tag', 'zs_qwen_cs1qa_qonly', '--batch', '4'],
}
RUN = 'smoke'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['zeroshot_qwen.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/zeroshot_qwen.py

In [ ]:
import argparse, json
from pathlib import Path
import torch
import kc_common as kc
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# zero-shot base LM: same question/code/answer text as CodeT5 templates, no fine-tuning
p = argparse.ArgumentParser()
p.add_argument("--model", default="Qwen/Qwen2.5-Coder-0.5B")
p.add_argument("--data", required=True)
p.add_argument("--split", default="test")
p.add_argument("--mode", choices=("full", "qonly"), required=True)
p.add_argument("--run-tag", required=True)
p.add_argument("--batch", type=int, default=16)
p.add_argument("--beams", type=int, default=10)
p.add_argument("--max-new", type=int, default=30)
p.add_argument("--max-code-tokens", type=int, default=350)
p.add_argument("--limit", type=int, default=0)
args = p.parse_args(ARGS)

data = Path(args.data)

In [ ]:
def read(ext):
    return (data / f"{args.split}.{ext}").read_text(encoding="utf-8").splitlines()

In [ ]:
C, Q, A = read("code"), read("question"), read("answer")
assert len(C) == len(Q) == len(A), (len(C), len(Q), len(A))
rows = [{"id": i, "code": c.strip(), "question": q.strip(), "answer": a.strip()} for i, (c, q, a) in enumerate(zip(C, Q, A))]
if args.limit:
    rows = rows[: args.limit]

tok = AutoTokenizer.from_pretrained(args.model)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=torch.bfloat16).to("cuda").eval()

In [ ]:
def clip(code):
    ids = tok(code, add_special_tokens=False)["input_ids"][: args.max_code_tokens]
    return tok.decode(ids)

def prompt(r):
    code = clip(r["code"]) if args.mode == "full" else ""
    return f"question: {r['question']}\ncode: {code}\nanswer:"

In [ ]:
enc = [tok(prompt(r), add_special_tokens=False)["input_ids"] for r in rows]
order = sorted(range(len(enc)), key=lambda i: -len(enc[i]))
preds = [None] * len(enc)
for s in range(0, len(order), args.batch):
    idx = order[s:s + args.batch]
    width = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), width), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for j, i in enumerate(idx):
        n = len(enc[i])
        ids[j, width - n:] = torch.tensor(enc[i]); mask[j, width - n:] = 1
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        gen = model.generate(input_ids=ids.cuda(), attention_mask=mask.cuda(), num_beams=args.beams,
                             max_new_tokens=args.max_new, do_sample=False, pad_token_id=tok.pad_token_id)
    for i, t in zip(idx, tok.batch_decode(gen[:, width:], skip_special_tokens=True)):
        preds[i] = next((l.strip() for l in t.split(chr(10)) if l.strip()), "")

metrics, per = kc.evaluate(preds, [r["answer"] for r in rows])
out = kc.EXP / "runs" / args.run_tag
out.mkdir(parents=True, exist_ok=True)
config = {"experiment": "ZEROSHOT", "model": args.model, "mode": args.mode, "data": str(data), "split": args.split,
          "decoding": f"beam{args.beams}", "max_code_tokens": args.max_code_tokens, "max_new_tokens": args.max_new,
          "fine_tuned": False, "n": len(rows)}
kc.write_json(out / "results.json", {"run_tag": args.run_tag, **config, args.split: metrics})
kc.write_predictions(out / f"{args.split}_predictions.jsonl", rows, preds, per, args.mode.upper(), [r["id"] for r in rows])
run = kc.wandb_init(out, args.run_tag, "ZEROSHOT", ["qwen25coder05b", args.mode, "zeroshot"], config, "inference")
records = kc.read_jsonl(out / f"{args.split}_predictions.jsonl")
kc.wandb_finalize(run, out, args.run_tag, metrics, records)
run.finish()
print("DONE", args.run_tag, json.dumps({k: round(metrics[k], 3) for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}), flush=True)

### Ringkasan hasil test (dibaca dari runs/<tag>/results.json)

In [ ]:
TAGS = ['smoke_zs_qwen_pythontest_full', 'zs_qwen_pythontest_full', 'zs_qwen_pythontest_qonly', 'zs_qwen_javatest_full', 'zs_qwen_javatest_qonly', 'zs_qwen_classevalqa_full', 'zs_qwen_classevalqa_qonly', 'zs_qwen_cs1qa_full', 'zs_qwen_cs1qa_qonly']
for tag in TAGS:
    for cand in (tag, "smoke_" + tag):
        f = EXP / "runs" / cand / "results.json"
        if f.exists():
            r = json.loads(f.read_text())
            t = r.get("test", {})
            print(cand, {k: round(v, 2) for k, v in t.items() if isinstance(v, (int, float))})
            break
    else:
        print(tag, "-> belum ada results.json")

## zeroshot_codet5.py

In [ ]:
VARIANTS = {
    'pythontest_full': [],
}
RUN = 'pythontest_full'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['zeroshot_codet5.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/zeroshot_codet5.py

In [ ]:
import argparse, json
from pathlib import Path
import torch
import kc_common as kc
from transformers import AutoTokenizer, T5ForConditionalGeneration

In [ ]:
# ablation: pretrained CodeT5-small with NO fine-tuning, same FULL template and beam-10 decoding as the fine-tuned runs
p = argparse.ArgumentParser()
p.add_argument("--model", default="Salesforce/codet5-small")
p.add_argument("--data", default=str(kc.DATA / "test"))
p.add_argument("--run-tag", default="zs_codet5_pythontest_full")
p.add_argument("--batch", type=int, default=16)
p.add_argument("--max-source", type=int, default=400)
p.add_argument("--max-new", type=int, default=30)
args = p.parse_args(ARGS)

data = Path(args.data)

In [ ]:
def read(ext):
    return (data / f"test.{ext}").read_text(encoding="utf-8").splitlines()

In [ ]:
C, Q, A = read("code"), read("question"), read("answer")
assert len(C) == len(Q) == len(A), (len(C), len(Q), len(A))
rows = [{"id": i, "code": c.strip(), "question": q.strip(), "answer": a.strip()} for i, (c, q, a) in enumerate(zip(C, Q, A))]

tok = AutoTokenizer.from_pretrained(args.model)
model = T5ForConditionalGeneration.from_pretrained(args.model).to("cuda").eval()
srcs = [kc.codet5_source(r["question"], r["code"], "full") for r in rows]
enc = tok(srcs, max_length=args.max_source, truncation=True)["input_ids"]
order = sorted(range(len(enc)), key=lambda i: -len(enc[i]))
preds = [None] * len(enc)
for s in range(0, len(order), args.batch):
    idx = order[s : s + args.batch]
    width = max(len(enc[i]) for i in idx)
    ids = torch.full((len(idx), width), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for j, i in enumerate(idx):
        ids[j, : len(enc[i])] = torch.tensor(enc[i]); mask[j, : len(enc[i])] = 1
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        gen = model.generate(input_ids=ids.cuda(), attention_mask=mask.cuda(), num_beams=10,
                             max_new_tokens=args.max_new, do_sample=False)
    for i, t in zip(idx, tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)):
        preds[i] = " ".join(t.split()).replace("|||", " ")

metrics, per = kc.evaluate(preds, [r["answer"] for r in rows])
out = kc.EXP / "runs" / args.run_tag
out.mkdir(parents=True, exist_ok=True)
config = {"experiment": "ABLATION", "model": args.model, "fine_tuned": False, "mode": "full", "data": str(data),
          "decoding": "beam10", "max_source": args.max_source, "max_new_tokens": args.max_new, "n": len(rows)}
kc.write_json(out / "results.json", {"run_tag": args.run_tag, **config, "test": metrics})
kc.write_predictions(out / "test_predictions.jsonl", rows, preds, per, "FULL", [r["id"] for r in rows])
run = kc.wandb_init(out, args.run_tag, "ABLATION", ["codet5small", "full", "zeroshot"], config, "inference")
records = kc.read_jsonl(out / "test_predictions.jsonl")
kc.wandb_finalize(run, out, args.run_tag, metrics, records)
run.finish()
print("DONE", args.run_tag, json.dumps({k: round(metrics[k], 3) for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}), flush=True)

## bertscore_xdomain.py
BERTScore prediksi zero-shot.

In [ ]:
VARIANTS = {
    'qwen': ['--set', 'zs_pythontest_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_pythontest_full/test_predictions.jsonl', '--set', 'zs_pythontest_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_pythontest_qonly/test_predictions.jsonl', '--set', 'zs_javatest_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_javatest_full/test_predictions.jsonl', '--set', 'zs_javatest_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_javatest_qonly/test_predictions.jsonl', '--set', 'zs_classevalqa_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_classevalqa_full/test_predictions.jsonl', '--set', 'zs_classevalqa_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_classevalqa_qonly/test_predictions.jsonl', '--set', 'zs_cs1qa_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_cs1qa_full/test_predictions.jsonl', '--set', 'zs_cs1qa_qonly=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_qwen_cs1qa_qonly/test_predictions.jsonl', '--out', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/bertscore_zeroshot_qwen.json'],
    'codet5': ['--set', 'zs_codet5_full=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/zs_codet5_pythontest_full/test_predictions.jsonl', '--out', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/bertscore_zs_codet5_python.json'],
}
RUN = 'codet5'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['bertscore_xdomain.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/bertscore_xdomain.py

In [ ]:
import argparse, json
from pathlib import Path
import bert_score

In [ ]:
E = "/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp"
p = argparse.ArgumentParser()
p.add_argument("--set", action="append", required=True, help="name=path to test_predictions.jsonl")
p.add_argument("--out", required=True)
args = p.parse_args(ARGS)
out = {}
for item in args.set:
    name, path = item.split("=", 1)
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    P, R, F = bert_score.score([r["prediction"] for r in rows], [r["reference"] for r in rows],
                               lang="en", batch_size=128, verbose=False, device="cuda")
    out[name] = {"path": path, "n": len(rows), "BERTScore_P": 100 * P.mean().item(), "BERTScore_R": 100 * R.mean().item(), "BERTScore_F1": 100 * F.mean().item()}
    print("DONE", name, json.dumps(out[name]), flush=True)
Path(args.out).write_text(json.dumps(out, indent=2), encoding="utf-8")
print("WROTE", args.out, flush=True)